In [5]:
!pip install -q pypdf sentence-transformers faiss-cpu gradio google-generativeai

In [9]:
from google.colab import files

uploaded = files.upload()


Saving Academic_Calendar_2026-27_FINAL.pdf to Academic_Calendar_2026-27_FINAL.pdf
Saving course-info-booklet-1.pdf to course-info-booklet-1.pdf
Saving grading.pdf to grading.pdf
Saving GuidelinesProcedureforawardingMinormastersprogramme_0.pdf to GuidelinesProcedureforawardingMinormastersprogramme_0.pdf
Saving IDDDP_Guidelines_2025.pdf to IDDDP_Guidelines_2025.pdf
Saving M.Sc.Rules.pdf to M.Sc.Rules.pdf
Saving ugrulebook.pdf to ugrulebook.pdf


In [10]:
import os
import pickle
import numpy as np
import faiss

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

In [16]:
documents = []

for filename in pdf_files:
    reader = PdfReader(filename)

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()

        if text and text.strip():
            documents.append({
                "text": text.strip(),
                "source": filename,
                "page": page_number
            })

print("Pages with extracted text:", len(documents))
print("\nSample text:\n")
print(documents[0]["text"][:500])

Pages with extracted text: 73

Sample text:

GUIDELINES     FOR     INTERDISCIPLINARY     DUAL     DEGREE     PROGRAMME  
(IDDDP)  
1) Minimum Eligibility Criteria:
a) Undergraduate (UG) students admitted to B.S., B.Tech. and DD (B.Tech. + M.Tech.)
programmes can apply for IDDDP at the end of sixth semester.
b) At the end of sixth semester, students must have CPI >= 7.5 and should not have any
FR/DR/DX/W grade in mandatory courses including NSO/NSS/NCC.
c) Through  IDDD  Programme,  students  can  apply  for  all  the  specializations  of 


In [17]:
def make_chunks(documents, chunk_size=500, overlap=100):
    chunks = []

    for doc in documents:
        words = doc["text"].split()
        start = 0

        while start < len(words):
            end = start + chunk_size
            chunk_text = " ".join(words[start:end])

            if chunk_text.strip():
                chunks.append({
                    "text": chunk_text,
                    "source": doc["source"],
                    "page": doc["page"]
                })

            start += chunk_size - overlap

    return chunks


chunks = make_chunks(documents)

print("Total chunks:", len(chunks))
print("\nSample chunk:\n")
print(chunks[0]["text"][:700])
print("\nSource:", chunks[0]["source"])
print("Page:", chunks[0]["page"])

Total chunks: 111

Sample chunk:

GUIDELINES FOR INTERDISCIPLINARY DUAL DEGREE PROGRAMME (IDDDP) 1) Minimum Eligibility Criteria: a) Undergraduate (UG) students admitted to B.S., B.Tech. and DD (B.Tech. + M.Tech.) programmes can apply for IDDDP at the end of sixth semester. b) At the end of sixth semester, students must have CPI >= 7.5 and should not have any FR/DR/DX/W grade in mandatory courses including NSO/NSS/NCC. c) Through IDDD Programme, students can apply for all the specializations of Dual Degree (DD) and M.Tech./ MBA/ M.Sc. programmes approved by the Academic Senate of IIT Bombay. d) Over and above minimum eligibility criteria [a-c], a DUGC/ DPGC may enforce additional eligibility and selection criteria [through A

Source: IDDDP_Guidelines_2025.pdf
Page: 1


In [18]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully


In [19]:
chunk_texts = [chunk["text"] for chunk in chunks]

embeddings = model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embedding shape: (111, 384)


In [20]:
embeddings = embeddings.astype("float32")
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("Vectors stored in FAISS:", index.ntotal)

Vectors stored in FAISS: 111


In [21]:
def retrieve_chunks(question, top_k=4):
    question_embedding = model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(question_embedding)

    scores, indices = index.search(question_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        chunk = chunks[idx]

        results.append({
            "text": chunk["text"],
            "source": chunk["source"],
            "page": chunk["page"],
            "score": float(score)
        })

    return results

In [22]:
question = "What is the eligibility criteria for IDDDP?"

results = retrieve_chunks(question)

for i, result in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Score:", round(result["score"], 4))
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Text:", result["text"][:500])


Result 1
Score: 0.6788
Source: ugrulebook.pdf
Page: 43
Text: III. will have a limit of two students in each unit of specialization except in the case of KCDH and CMInDS (8 TA + 22 RAP for each). g) The selection and entry of all candidates in IDDDP will remain provisional till the successful completion of B.S./ B.Tech. curriculum by the end of 8th semester. The payment of TAship

Result 2
Score: 0.655
Source: IDDDP_Guidelines_2025.pdf
Page: 1
Text: GUIDELINES FOR INTERDISCIPLINARY DUAL DEGREE PROGRAMME (IDDDP) 1) Minimum Eligibility Criteria: a) Undergraduate (UG) students admitted to B.S., B.Tech. and DD (B.Tech. + M.Tech.) programmes can apply for IDDDP at the end of sixth semester. b) At the end of sixth semester, students must have CPI >= 7.5 and should not have any FR/DR/DX/W grade in mandatory courses including NSO/NSS/NCC. c) Through IDDD Programme, students can apply for all the specializations of Dual Degree (DD) and M.Tech./ MBA/

Result 3
Score: 0.6417
Source: IDDDP_Guideli

In [26]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 984.2/984.2 kB 26.4 MB/s eta 0:00:00


In [27]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

print("Gemini connected successfully!")

Gemini connected successfully!


In [28]:
def answer_question(question, top_k=4, threshold=0.35):
    results = retrieve_chunks(question, top_k=top_k)

    if not results or results[0]["score"] < threshold:
        return "I don't know based on the available documents.", []

    context_parts = []

    for i, result in enumerate(results, start=1):
        context_parts.append(
            f"""Source {i}: {result['source']}, Page {result['page']}
{result['text']}"""
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
You are IITB Insti-Assist, an academic assistant for IIT Bombay.

Answer the question using only the context below.

Rules:
- Do not use outside knowledge.
- Do not guess.
- If the answer is not clearly supported by the context, reply exactly:
  "I don't know based on the available documents."
- Keep the answer clear and concise.

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )

    answer = response.text.strip()

    sources = []
    seen = set()

    for result in results:
        source_key = (result["source"], result["page"])

        if source_key not in seen:
            sources.append({
                "source": result["source"],
                "page": result["page"],
                "score": result["score"],
                "text": result["text"]
            })
            seen.add(source_key)

    return answer, sources

In [29]:
question = "What is the minimum eligibility criteria for IDDDP?"

answer, sources = answer_question(question)

print("Answer:\n")
print(answer)

print("\nSources:\n")

for source in sources:
    print(
        f"- {source['source']}, Page {source['page']} "
        f"(Score: {source['score']:.3f})"
    )

Answer:

Based on the provided documents, the minimum eligibility criteria for the Interdisciplinary Dual Degree Programme (IDDDP) are:

1. **Eligible Programmes:** Undergraduate (UG) students admitted to B.S., B.Tech., and DD (B.Tech. + M.Tech.) programmes can apply at the end of their sixth semester.
2. **Academic Record:** At the end of the sixth semester, students must have a CPI $\ge$ 7.5 and must not have any FR/DR/DX/W grade in mandatory courses, including NSO/NSS/NCC.
3. **Ineligibility:** Students admitted to the B.S. programme through the Maths Olympiad are not eligible.
4. **Additional Criteria:** Individual DUGC/DPGC may enforce additional eligibility and selection criteria over and above these minimum requirements.

Sources:

- ugrulebook.pdf, Page 43 (Score: 0.650)
- IDDDP_Guidelines_2025.pdf, Page 1 (Score: 0.642)
- IDDDP_Guidelines_2025.pdf, Page 4 (Score: 0.633)


In [30]:
question = "Who will win the next IPL season?"

answer, sources = answer_question(question)

print("Answer:\n")
print(answer)

print("\nTop retrieval score:")
if sources:
    print(round(sources[0]["score"], 3))
else:
    print("No sources used")

Answer:

I don't know based on the available documents.

Top retrieval score:
No sources used


In [31]:
import gradio as gr


def chat_with_sources(question):
    if not question.strip():
        return "Please enter a question.", ""

    answer, sources = answer_question(question)

    if not sources:
        return answer, "No supporting source found."

    source_lines = []

    for i, source in enumerate(sources, start=1):
        source_lines.append(
            f"{i}. {source['source']} — Page {source['page']} "
            f"(score: {source['score']:.3f})"
        )

    source_text = "\n".join(source_lines)

    return answer, source_text


with gr.Blocks(title="IITB Insti-Assist") as demo:
    gr.Markdown(
        """
        # IITB Insti-Assist
        Ask questions about IIT Bombay academic rules and programmes.

        The assistant answers only from the uploaded documents.
        """
    )

    question_box = gr.Textbox(
        label="Your question",
        placeholder="Example: What is the eligibility criteria for IDDDP?",
        lines=2
    )

    ask_button = gr.Button("Ask")

    answer_box = gr.Textbox(
        label="Answer",
        lines=8
    )

    source_box = gr.Textbox(
        label="Sources used",
        lines=5
    )

    ask_button.click(
        fn=chat_with_sources,
        inputs=question_box,
        outputs=[answer_box, source_box]
    )

    question_box.submit(
        fn=chat_with_sources,
        inputs=question_box,
        outputs=[answer_box, source_box]
    )


demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6f657d730c1cdc175f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
